# ✨ VibeCast — ElevenLabs Colab Launcher

This short notebook uploads and launches the complete VibeCast Streamlit app. It uses Groq for commentary and ElevenLabs v4 for expressive MP3 narration.

Before launching, add these Colab Secrets:
- `GROQ_API_KEY`
- `ELEVENLABS_API_KEY`
- `ELEVENLABS_VOICE_ID` — one voice ID for the one on-screen VibeCast character.

The detailed instructor notebook explains the one-character/one-voice setup, five vibe audio tags, tests and troubleshooting.

**Account warning:** If ElevenLabs reports `detected_unusual_activity` and Free Tier access disabled, the TTS API remains blocked. A single voice ID does not bypass an account-level restriction.

## 1. Upload and extract the project ZIP

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files, userdata
import os, shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise FileNotFoundError('Upload VibeCast_SingleCharacter_ElevenLabs_Project.zip.')

project_dir = Path('/content/VibeCast_Project')
old_process = globals().get('process')
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try: old_process.wait(timeout=5)
    except Exception: old_process.kill()
if project_dir.exists(): shutil.rmtree(project_dir)
with ZipFile(zip_names[0]) as archive: archive.extractall('/content')
if not (project_dir / 'app.py').exists():
    raise FileNotFoundError('ZIP must contain VibeCast_Project/app.py.')
os.chdir(project_dir)
print('Project ready:', project_dir)

## 2. Install dependencies and load secrets

In [ ]:
%pip install -q -r requirements.txt

for name in (
    'GROQ_API_KEY', 'ELEVENLABS_API_KEY', 'ELEVENLABS_VOICE_ID'
):
    try: value = userdata.get(name) or ''
    except Exception: value = ''
    if value: os.environ[name] = value
    else: os.environ.pop(name, None)

from dotenv import set_key, load_dotenv
# Write configured secrets only into the temporary project .env. Never print them.
env_path = project_dir / '.env'
if not env_path.exists(): env_path.write_text('', encoding='utf-8')
for name in (
    'GROQ_API_KEY', 'ELEVENLABS_API_KEY', 'ELEVENLABS_VOICE_ID'
):
    if os.getenv(name): set_key(str(env_path), name, os.getenv(name))
set_key(str(env_path), 'ELEVENLABS_TTS_MODEL', 'eleven_v4')
set_key(str(env_path), 'ELEVENLABS_OUTPUT_FORMAT', 'mp3_44100_128')
load_dotenv(env_path, override=True)
print('Groq configured:', bool(os.getenv('GROQ_API_KEY')))
print('ElevenLabs key configured:', bool(os.getenv('ELEVENLABS_API_KEY')))
print('One VibeCast character voice ID configured:', bool(os.getenv('ELEVENLABS_VOICE_ID')))


## 3. Launch Streamlit

Streamlit runs on port 8501. Keep the Colab runtime active while using the website. The iframe method is used instead of the older window-opening helper.

In [ ]:
import subprocess, sys, time
from google.colab import output

log_path = Path('/content/vibecast_streamlit.log')
log_file = open(log_path, 'w')
process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501', '--server.address', '0.0.0.0',
     '--server.headless', 'true', '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    cwd=str(project_dir), stdout=log_file, stderr=subprocess.STDOUT
)
time.sleep(5)
if process.poll() is not None:
    print(log_path.read_text(errors='ignore')[-6000:])
    raise RuntimeError('Streamlit did not start. Check the log above.')
print('VibeCast is running.')
output.serve_kernel_port_as_iframe(8501, height=900)

## Troubleshooting

- If `ELEVENLABS_VOICE_ID` is missing, add one API-accessible voice ID in Colab Secrets.
- If ElevenLabs returns 401 for an invalid key, verify the key and its Text to Speech permission.
- If it returns `detected_unusual_activity`, Free Tier API access is disabled on the account. A single voice ID does not bypass the restriction; contact ElevenLabs support or use an eligible paid subscription.
- If it returns 403 `feature_not_available` from Voice Design, do not create voices through the API. This project does not require voice creation.
- If it returns 429, check the monthly credits and concurrency limit.
- If the website shows an old version, restart the runtime and re-upload the latest ZIP.
- For complete explanations and emotion comparisons, use `VibeCast_Commentary_Instructor.ipynb`.
